# Complex maze environment

The Colab version of this code can be used with the below commands. The code can also be downloaded locally and run as a Python Jupyter notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Useful imports:

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from enum import Enum
import random
import gymnasium as gym
from gymnasium import spaces
from gymnasium.envs.registration import register
from gymnasium.utils.env_checker import check_env
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="gymnasium")

### Creation of the complex maze environment

The source of the code is built from the one of <a href="https://gymnasium.farama.org/tutorials/gymnasium_basics/environment_creation/">Gymnasium</a>. Also refer to "Writing your own environment" from <a href="https://sarl-plus.github.io/rl-bootcamp-setup/">RL Bootcamp - Participant Primer</a>.

In [ ]:
class Actions(Enum):
    RIGHT = 0
    UP = 1
    LEFT = 2
    DOWN = 3

class ComplexGridWorldEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array", "rgb_array_list"], "render_fps": 4}

    def __init__(self, hole_map, mountains_map, render_mode=None, size=5, max_steps=100):
        self.size = size # The size of the square grid
        self.max_steps = max_steps
        self.current_step = 0

        self._agent_location = np.array([-1, -1], dtype=np.int64)
        self._target_location = np.array([-1, -1], dtype=np.int64)

        # Add the hole and mountain map
        if hole_map is None:
           self.holes = np.zeros((size, size), dtype=np.int64)
        else:
           self.holes = np.array(hole_map, dtype=np.int64)
           self.size = self.holes.shape[0]  # Sync size with hole_map dimensions
        if mountains_map is None:
           self.mountains = np.zeros((size, size), dtype=np.int64)
        else:
           self.mountains = np.array(mountains_map, dtype=np.int64)
           self.size = self.mountains.shape[0]  # Sync size with mountains_map dimensions

        # Updated Observation Space to include the new elements
        # Each location is encoded as an element of {0, ..., 'size'}^2, i.e. MultiDiscrete([size, size])
        self.observation_space = spaces.Dict({
            "agent": spaces.Box(0, size-1, shape=(2,), dtype=np.int64),
            "target": spaces.Box(0, size - 1, shape=(2,), dtype=np.int64),
            "holes": spaces.Box(0, 1, shape=(size, size), dtype=np.int64),
            "mountains": spaces.Box(0, 1, shape=(size, size), dtype=np.int64)
        })

        # We have 4 actions, corresponding to "right", "up", "left", "down"
        self.action_space = spaces.Discrete(4)

        """
        The following dictionary maps abstract actions from 'self.action_space' to
        the direction we will walk in if that action is taken.
        i.e. 0 corresponds to "right", 1 to "up" etc.
        Uses NumPy [row, col] convention where row 0 is at the top.
        """
        self._action_to_direction = {
            Actions.RIGHT.value: np.array([0, 1], dtype=np.int64),
            Actions.UP.value: np.array([-1, 0], dtype=np.int64),
            Actions.LEFT.value: np.array([0, -1], dtype=np.int64),
            Actions.DOWN.value: np.array([1, 0], dtype=np.int64),
        }

        assert render_mode is None or render_mode in self.metadata["render_modes"]
        self.render_mode = render_mode

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0

        # Ensure valid target location relative to current grid size
        self._target_location = np.array([self.size - 1, self.size - 1], dtype=np.int64)

        while True:
            pos = self.np_random.integers(0, self.size, size=2, dtype=np.int64)
            if self.holes[pos[0], pos[1]] == 0:
                self._agent_location = pos
                break

        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "rgb_array":
            self._render_frame()

        return observation, info

    def step(self, action):
        self.current_step += 1

        direction = self._action_to_direction[action]
        # We use 'np.clip' to make sure we don't leave the grid
        new_position = np.clip(
            self._agent_location + direction, 0, self.size - 1
        )

        # prevent movement into holes
        if self.holes[new_position[0], new_position[1]] == 0:
          self._agent_location = new_position

        # An episode is done iff the agent has reached the target
        terminated = np.array_equal(self._agent_location, self._target_location)
        truncated = self.current_step >= self.max_steps
        # Assign different rewards dependings on the new relief
        if terminated:
            reward = 10.0  # Success reward
        elif self.holes[new_position[0], new_position[1]] == 1:
            reward = -5.0  # Failure reward
        elif self.mountains[new_position[0], new_position[1]] == 1:
            reward = -0.5  # Cost reward
        else:
          reward = 0.0

        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "rgb_array":
            self._render_frame()

        return observation, reward, terminated, truncated, info

    def _get_obs(self):
        return {
            "agent": self._agent_location,
            "target": self._target_location,
            "holes": self.holes,
            "mountains": self.mountains,
        }

    def _get_info(self):
        return {
            "distance": np.linalg.norm(self._agent_location - self._target_location, ord=1)
        }

    def render(self):
        if self.render_mode in ["rgb_array", "rgb_array_list"]:
            return self._render_frame()
        return None

    def _render_frame(self):
        size = self.size
        # Create a blank RGB canvas (light blue)
        canvas = np.ones((size, size, 3), dtype=np.uint8) * [0, 255, 255]

        # Draw holes (blue)
        canvas[self.holes == 1] = [0, 0, 225]

        # Draw mountains (brown)
        canvas[self.mountains == 1] = [139, 115, 85]

        # Draw target (green)
        canvas[self._target_location[0], self._target_location[1]] = [0, 128, 0]

        return canvas.astype(np.uint8)

    def close(self):
        pass

In [ ]:
# Register Environment of size 5
gym.register(
    id="ComplexGridWorld-v1",
    entry_point="__main__:ComplexGridWorldEnv",
)

hole_map = np.array([
    [0,1,0,0,0],
    [0,1,1,1,0],
    [0,0,0,0,0],
    [0,1,1,1,0],
    [0,0,0,0,0]
])

mountains_map = np.array([
    [0,0,0,1,0],
    [0,0,0,0,0],
    [0,0,1,0,0],
    [0,0,0,0,0],
    [0,0,0,0,0]
])

env = gym.make(
    "ComplexGridWorld-v1",
    hole_map=hole_map,
    mountains_map=mountains_map,
    render_mode="rgb_array_list",
    size=5
)

frame = env.unwrapped._render_frame() # remove the layers to access base methods and properties of the environment
plt.imshow(frame)
plt.title("Green=Goal, Blue=Hole, Brown=Mountain")
plt.axis('off')
plt.show()

In [ ]:
check_env(ComplexGridWorldEnv(hole_map, mountains_map))